In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
# 1. 读取清洗后的数据（路径请确认）
df = pd.read_csv("../data/cleaned_data.csv")

In [2]:
# 2. 只保留购买行为
buy_df = df[df["behavior"] == "buy"].copy()
print(f"购买记录总数: {len(buy_df)}")
print(f"购买用户数: {buy_df['user_id'].nunique()}")
print(f"商品种类数: {buy_df['item_id'].nunique()}")

# 3. 过滤低频商品：只保留被足够多用户购买的商品
# 先统计每个商品的购买用户数
item_user_cnt = buy_df.groupby('item_id')['user_id'].nunique()
# 设定阈值：至少被 200 个不同用户购买（可根据数据调整）
min_user_per_item = 200
popular_items = item_user_cnt[item_user_cnt >= min_user_per_item].index
buy_df = buy_df[buy_df['item_id'].isin(popular_items)]
print(f"过滤后商品数: {len(popular_items)}")
print(f"过滤后购买记录数: {len(buy_df)}")

# 4. 抽样用户（防止内存爆炸）
# 抽样 5 万用户（如果电脑内存大，可增加到 10 万）
all_users = buy_df['user_id'].unique()
sample_users = np.random.choice(all_users, size=min(50000, len(all_users)), replace=False)
buy_df = buy_df[buy_df['user_id'].isin(sample_users)]
print(f"抽样用户数: {len(sample_users)}")
print(f"抽样后购买记录数: {len(buy_df)}")

# 5. 构造交易列表：每个用户购买的商品列表
transactions = buy_df.groupby('user_id')['item_id'].apply(lambda x: list(set(x))).tolist()
# 过滤掉空交易（理论上不会有，但安全起见）
transactions = [t for t in transactions if len(t) > 0]
print(f"有效交易数: {len(transactions)}")

# 6. 编码（TransactionEncoder）
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)
# 列名转为字符串（避免后续报错）
df_trans.columns = [str(col) for col in df_trans.columns]
print(f"编码后的矩阵形状: {df_trans.shape}")

# 7. 动态设置最小支持度
# 我们希望项集至少出现在 50 个用户中（可调整）
min_support = 50 / len(transactions)
print(f"最小支持度阈值: {min_support:.5f} (即至少 {int(min_support * len(transactions))} 个用户)")

# 8. 运行 FP-Growth
frequent_items = fpgrowth(df_trans, min_support=min_support, use_colnames=True)

if frequent_items.empty:
    print("警告：没有找到频繁项集，尝试降低支持度到 30 个用户")
    min_support = 30 / len(transactions)
    frequent_items = fpgrowth(df_trans, min_support=min_support, use_colnames=True)

if frequent_items.empty:
    print("仍然为空，数据可能过于稀疏。尝试进一步降低支持度到 20 个用户")
    min_support = 20 / len(transactions)
    frequent_items = fpgrowth(df_trans, min_support=min_support, use_colnames=True)

if frequent_items.empty:
    print("错误：无法找到任何频繁项集。请检查数据或降低过滤门槛。")
else:
    print(f"找到 {len(frequent_items)} 个频繁项集")
    # 9. 生成关联规则
    rules = association_rules(frequent_items, metric="confidence", min_threshold=0.2)
    # 按提升度排序，取前 20
    rules = rules.sort_values("lift", ascending=False).head(20)
    # 保存结果
    import os
    os.makedirs("../result", exist_ok=True)
    rules.to_csv("../result/fpgrowth_rules.csv", index=False)
    print("✅ 关联规则挖掘成功！结果已保存到 ../result/fpgrowth_rules.csv")
    print("\n前10条规则（按提升度降序）：")
    print(rules[["antecedents", "consequents", "confidence", "lift"]].head(10))

购买记录总数: 1998944
购买用户数: 670370
商品种类数: 635872
过滤后商品数: 162
过滤后购买记录数: 49110
抽样用户数: 43197
抽样后购买记录数: 49110
有效交易数: 43197
编码后的矩阵形状: (43197, 162)
最小支持度阈值: 0.00116 (即至少 50 个用户)
找到 166 个频繁项集
✅ 关联规则挖掘成功！结果已保存到 ../result/fpgrowth_rules.csv

前10条规则（按提升度降序）：
            antecedents           consequents  confidence       lift
0  frozenset({1042152})  frozenset({5062984})    0.297674  38.043319
1  frozenset({1042152})  frozenset({5051027})    0.237209  35.091542
2  frozenset({5051027})  frozenset({5062984})    0.267123  34.138830
3  frozenset({5062984})  frozenset({5051027})    0.230769  34.138830
